# GA-Planes NeRF (JAX) -- Multiresolution

This notebook implements **GA-Planes** for NeRF in pure JAX, with the full
multiresolution scheme described in the paper.

## Architecture

Given base resolutions `[r1, r2, r3]`, channel dimensions `[d1, d2, d3]`,
and multiresolution upsampling factors `[m1, m2, ..., mK]`:

- **Line features** (per axis x/y/z, per resolution level `k`):
  shape `(d1, mk * r1)` -- K copies per axis, 3K grids total.
- **Plane features** (per canonical plane xy/yz/zx, per level `k`):
  shape `(d2, mk * r2, mk * r2)` -- K copies per plane, 3K grids total.
- **Volume feature** (single resolution, no multiresolution copies):
  shape `(d3, r3, r3, r3)`.

At each resolution level `k`, features are combined via the GA-Planes
*concatenate* operation:
```
level_k = [line_x_k * line_y_k * line_z_k,
           plane_xy_k * line_z_k,
           plane_yz_k * line_x_k,
           plane_zx_k * line_y_k]          # 4 * d1 channels
```
All levels are concatenated along with the volume feature and passed to
a 2-layer ReLU decoder:
```
decoder_input = [level_1, ..., level_K, volume]   # K*4*d1 + d3 channels
```

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import pickle
import sys
sys.path.append('../')
from helpers import *

import jax
from jax import random, grad, jit, vmap
import jax.numpy as np
from jax.example_libraries import optimizers

from livelossplot import PlotLosses
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import numpy as onp

jax.devices()
jax.default_backend()

rng = random.PRNGKey(0)

# Load Data

In [ ]:
filename = 'lego_400.npz'
if not os.path.exists(filename):
    !gdown --id 108jNfjPITTsTA0lE6Kpg7Ei53BUVL-4n  # Lego

data = np.load(filename)
images = data['images']
poses = data['poses']
focal = float(data['focal'])
H, W = images.shape[1:3]

images, val_images, test_images = np.split(images[...,:3], [100,107], axis=0)
poses, val_poses, test_poses = np.split(poses, [100,107], axis=0)

print(val_images.shape, test_images.shape, focal)
plt.imshow(test_images[0,...])
plt.show()

# Rendering Functions

In [ ]:
def get_rays(H, W, focal, c2w):
    """Generate camera rays for a given pose."""
    i, j = np.meshgrid(np.arange(W), np.arange(H), indexing='xy')
    dirs = np.stack([(i-W*.5)/focal, -(j-H*.5)/focal, -np.ones_like(i)], -1)
    rays_d = np.sum(dirs[..., np.newaxis, :] * c2w[:3,:3], -1)
    rays_o = np.broadcast_to(c2w[:3,-1], rays_d.shape)
    return np.stack([rays_o, rays_d], 0)

get_rays = jit(get_rays, static_argnums=(0, 1, 2,))

training_rays = np.stack([get_rays(H,W,focal,pose) for pose in poses], 1)
training_data = np.concatenate([training_rays, images[None]])
training_data = np.moveaxis(training_data, 0, -2)
training_data = onp.array(np.reshape(training_data, [-1, 3, 3]))
onp.random.shuffle(training_data)
training_data = np.array(training_data)

In [ ]:
DENSITY_BIAS = -3.0          # softplus(raw_sigma + bias); -3 starts nearly transparent but trainable
WHITE_BACKGROUND = False     # Lego data shown in this notebook has black background
MASK_OUTSIDE_BBOX = True     # avoid learning density on clipped boundary samples


def render_rays(params, key, rays, near, far, N_samples,
                scene_bbox, n_levels, rand=False, allret=False):
    rays_o, rays_d = rays
    bbox_min, bbox_max = scene_bbox

    # ---- sample points along each ray ----
    z_vals = np.linspace(near, far, N_samples)
    if rand:
        z_vals = z_vals + random.uniform(
            key, shape=list(rays_o.shape[:-1]) + [N_samples]
        ) * (far - near) / N_samples

    pts = rays_o[..., None, :] + rays_d[..., None, :] * z_vals[..., :, None]

    # ---- evaluate GA-Planes on every query point ----
    pts_flat = np.reshape(pts, [-1, 3])

    if MASK_OUTSIDE_BBOX:
        inside_flat = np.all((pts_flat >= bbox_min) & (pts_flat <= bbox_max), axis=-1)
    else:
        inside_flat = np.ones((pts_flat.shape[0],), dtype=bool)

    raw = gaplanes_apply(params, pts_flat, scene_bbox, n_levels)
    raw = np.reshape(raw, list(pts.shape[:-1]) + [4])
    inside = np.reshape(inside_flat, pts.shape[:-1])

    # ---- decode rgb + sigma ----
    rgb = jax.nn.sigmoid(raw[..., :3])

    sigma_a = jax.nn.softplus(raw[..., 3] + DENSITY_BIAS)
    sigma_a = sigma_a * inside

    # ---- volume rendering ----
    dists = np.concatenate([
        z_vals[..., 1:] - z_vals[..., :-1],
        z_vals[..., -1:] - z_vals[..., -2:-1],
    ], axis=-1)

    # Convert parameter-space z intervals into world-space distances.
    dists = dists * np.linalg.norm(rays_d[..., None, :], axis=-1)

    alpha = 1.0 - np.exp(-sigma_a * dists)

    trans = np.concatenate([
        np.ones_like(alpha[..., :1]),
        1.0 - alpha + 1e-10,
    ], axis=-1)
    trans = np.cumprod(trans, axis=-1)[..., :-1]
    weights = alpha * trans

    rgb_map = np.sum(weights[..., None] * rgb, axis=-2)
    acc_map = np.sum(weights, axis=-1)

    if WHITE_BACKGROUND:
        rgb_map = rgb_map + (1.0 - acc_map)[..., None]

    if not allret:
        return rgb_map

    depth_map = np.sum(weights * z_vals, axis=-1)
    return rgb_map, depth_map, acc_map


def render_fn_inner(params, key, rays, scene_bbox, n_levels, rand, allret):
    return render_rays(params, key, rays,
                       near=2., far=6., N_samples=N_samples,
                       scene_bbox=scene_bbox, n_levels=n_levels,
                       rand=rand, allret=allret)

render_fn_inner = jit(render_fn_inner, static_argnums=(4, 5, 6,))


def render_fn(params, key, rays, scene_bbox, n_levels, rand):
    """Render a full image in row chunks to avoid OOM."""
    chunk = 5
    for i in range(0, rays.shape[1], chunk):
        out = render_fn_inner(params, key, rays[:, i:i+chunk],
                              scene_bbox, n_levels, rand, True)
        if i == 0:
            rets = out
        else:
            rets = [np.concatenate([a, b], 0) for a, b in zip(rets, out)]
    return rets


# GA-Planes Model Definition

Pure-JAX implementation of multiresolution `GAPlanes3D`.

**Multiresolution scheme** (from the paper, lines 724-731):

Given base resolutions `[r1, r2, r3]`, channel dims `[d1, d2, d3]`, and
upsampling factors `[m1, m2, ..., mK]`:

| Grid type | Copy k shape | # copies |
|-----------|-------------|----------|
| Lines g1,g2,g3 | `(d1, mk*r1)` | 3 x K |
| Planes g12,g23,g13 | `(d2, mk*r2, mk*r2)` | 3 x K |
| Volume g123 | `(d3, r3, r3, r3)` | 1 (no multires) |

In [ ]:
def interp_1d(feature, coord):
    """
    1-D linear interpolation of a line feature.

    Args:
        feature : (feat_dim, resolution)
        coord   : (N,) values in [-1, 1]
    Returns:
        (N, feat_dim)
    """
    res = feature.shape[1]
    idx_f = (coord + 1.0) * 0.5 * (res - 1)
    idx0 = np.floor(idx_f).astype(np.int32)
    idx1 = idx0 + 1
    idx0 = np.clip(idx0, 0, res - 1)
    idx1 = np.clip(idx1, 0, res - 1)
    w = idx_f - np.floor(idx_f)
    f0 = feature[:, idx0]
    f1 = feature[:, idx1]
    return ((1.0 - w)[None] * f0 + w[None] * f1).T   # (N, feat_dim)


def interp_2d(feature, ca, cb):
    """
    2-D bilinear interpolation of a plane feature.

    Args:
        feature : (feat_dim, res, res)
        ca, cb  : (N,) each in [-1, 1]
    Returns:
        (N, feat_dim)
    """
    res = feature.shape[1]
    a = (ca + 1.0) * 0.5 * (res - 1)
    b = (cb + 1.0) * 0.5 * (res - 1)
    a0 = np.clip(np.floor(a).astype(np.int32), 0, res - 1)
    a1 = np.clip(a0 + 1, 0, res - 1)
    b0 = np.clip(np.floor(b).astype(np.int32), 0, res - 1)
    b1 = np.clip(b0 + 1, 0, res - 1)
    wa = a - np.floor(a)
    wb = b - np.floor(b)
    w00 = (1 - wa) * (1 - wb)
    w01 = (1 - wa) * wb
    w10 = wa * (1 - wb)
    w11 = wa * wb
    f00 = feature[:, a0, b0]
    f01 = feature[:, a0, b1]
    f10 = feature[:, a1, b0]
    f11 = feature[:, a1, b1]
    return (w00[None]*f00 + w01[None]*f01 + w10[None]*f10 + w11[None]*f11).T


def interp_3d(feature, ca, cb, cc):
    """
    3-D trilinear interpolation of a volume feature.

    Args:
        feature    : (feat_dim, res, res, res)
        ca, cb, cc : (N,) each in [-1, 1]
    Returns:
        (N, feat_dim)
    """
    res = feature.shape[1]
    a = (ca + 1.0) * 0.5 * (res - 1)
    b = (cb + 1.0) * 0.5 * (res - 1)
    c = (cc + 1.0) * 0.5 * (res - 1)
    a0 = np.clip(np.floor(a).astype(np.int32), 0, res - 1)
    a1 = np.clip(a0 + 1, 0, res - 1)
    b0 = np.clip(np.floor(b).astype(np.int32), 0, res - 1)
    b1 = np.clip(b0 + 1, 0, res - 1)
    c0 = np.clip(np.floor(c).astype(np.int32), 0, res - 1)
    c1 = np.clip(c0 + 1, 0, res - 1)
    wa = a - np.floor(a)
    wb = b - np.floor(b)
    wc = c - np.floor(c)
    # 8 corners
    f000 = feature[:, a0, b0, c0]
    f001 = feature[:, a0, b0, c1]
    f010 = feature[:, a0, b1, c0]
    f011 = feature[:, a0, b1, c1]
    f100 = feature[:, a1, b0, c0]
    f101 = feature[:, a1, b0, c1]
    f110 = feature[:, a1, b1, c0]
    f111 = feature[:, a1, b1, c1]
    w000 = (1 - wa) * (1 - wb) * (1 - wc)
    w001 = (1 - wa) * (1 - wb) * wc
    w010 = (1 - wa) * wb * (1 - wc)
    w011 = (1 - wa) * wb * wc
    w100 = wa * (1 - wb) * (1 - wc)
    w101 = wa * (1 - wb) * wc
    w110 = wa * wb * (1 - wc)
    w111 = wa * wb * wc
    return (w000[None]*f000 + w001[None]*f001 + w010[None]*f010 + w011[None]*f011 +
            w100[None]*f100 + w101[None]*f101 + w110[None]*f110 + w111[None]*f111).T

In [ ]:
def init_gaplanes(key, r1, r2, r3, d1, d2, d3,
                  multires_factors, hidden_dim, output_dim=4):
    assert d1 == d2, (
        f'Line and plane channel dims must match for element-wise gating '
        f'in concatenate mode, got d1={d1}, d2={d2}'
    )
    K = len(multires_factors)
    decoder_in = K * 4 * d1 + d3

    n_keys = 3*K + 3*K + 1 + 4  # lines + planes + volume + decoder
    keys = random.split(key, n_keys)
    ki = 0  # key index counter

    params = {}

    # --- multiresolution line features ---
    for k, mk in enumerate(multires_factors):
        res = int(mk * r1)
        params[f'line_x_{k}'] = random.normal(keys[ki], (d1, res)) * 0.1;  ki += 1
        params[f'line_y_{k}'] = random.normal(keys[ki], (d1, res)) * 0.1;  ki += 1
        params[f'line_z_{k}'] = random.normal(keys[ki], (d1, res)) * 0.1;  ki += 1

    # --- multiresolution plane features ---
    for k, mk in enumerate(multires_factors):
        res = int(mk * r2)
        params[f'plane_xy_{k}'] = random.normal(keys[ki], (d2, res, res)) * 0.01;  ki += 1
        params[f'plane_yz_{k}'] = random.normal(keys[ki], (d2, res, res)) * 0.01;  ki += 1
        params[f'plane_zx_{k}'] = random.normal(keys[ki], (d2, res, res)) * 0.01;  ki += 1

    # --- single-resolution volume feature ---
    params['volume'] = random.normal(keys[ki], (d3, r3, r3, r3)) * 0.001;  ki += 1

    # --- decoder MLP (2 layers) ---
    params['W1'] = random.normal(keys[ki], (decoder_in, hidden_dim)) * 0.01;  ki += 1
    params['b1'] = np.zeros(hidden_dim);  ki += 1
    params['W2'] = random.normal(keys[ki], (hidden_dim, output_dim)) * 0.01;  ki += 1
    params['b2'] = np.zeros(output_dim)

    return params


def count_params(params):
    return sum(v.size for v in jax.tree.leaves(params))


def describe_grids(params, multires_factors, r1, r2, r3, d1, d2, d3):
    """Print a summary table of all grids and their shapes."""
    total = 0
    print(f'{"Grid":>20s}  {"Shape":>25s}  {"Params":>10s}')
    print('-' * 60)
    for k, mk in enumerate(multires_factors):
        for axis in ['x', 'y', 'z']:
            key = f'line_{axis}_{k}'
            s = params[key].shape
            n = params[key].size
            total += n
            print(f'{key:>20s}  {str(s):>25s}  {n:>10,d}')
    for k, mk in enumerate(multires_factors):
        for plane in ['xy', 'yz', 'zx']:
            key = f'plane_{plane}_{k}'
            s = params[key].shape
            n = params[key].size
            total += n
            print(f'{key:>20s}  {str(s):>25s}  {n:>10,d}')
    s = params['volume'].shape
    n = params['volume'].size
    total += n
    print(f'{"volume":>20s}  {str(s):>25s}  {n:>10,d}')
    print('-' * 60)
    dec = sum(params[k].size for k in ['W1','b1','W2','b2'])
    total += dec
    print(f'{"decoder":>20s}  {"":>25s}  {dec:>10,d}')
    print(f'{"TOTAL":>20s}  {"":>25s}  {total:>10,d}')

In [ ]:
def gaplanes_apply(params, pts, scene_bbox, n_levels):
    bbox_min, bbox_max = scene_bbox
    pts_n = 2.0 * (pts - bbox_min) / (bbox_max - bbox_min) - 1.0
    pts_n = np.clip(pts_n, -1.0, 1.0)

    x, y, z = pts_n[:, 0], pts_n[:, 1], pts_n[:, 2]

    level_features = []
    for k in range(n_levels):
        # ---- interpolate line features at level k ----
        feat_x = interp_1d(params[f'line_x_{k}'], x)
        feat_y = interp_1d(params[f'line_y_{k}'], y)
        feat_z = interp_1d(params[f'line_z_{k}'], z)

        # ---- interpolate plane features at level k ----
        feat_xy = interp_2d(params[f'plane_xy_{k}'], x, y)
        feat_yz = interp_2d(params[f'plane_yz_{k}'], y, z)
        feat_zx = interp_2d(params[f'plane_zx_{k}'], z, x)

        # ---- GA-Planes combination (concatenate operation) ----
        combined_lines = feat_x * feat_y * feat_z
        gated_xy       = feat_xy * feat_z
        gated_yz       = feat_yz * feat_x
        gated_zx       = feat_zx * feat_y

        level_features.append(
            np.concatenate([combined_lines, gated_xy, gated_yz, gated_zx], axis=-1)
        )  # (N, 4 * d1)

    # ---- single-resolution volume feature ----
    feat_vol = interp_3d(params['volume'], x, y, z)  # (N, d3)

    # ---- concatenate all levels + volume ----
    features = np.concatenate(level_features + [feat_vol], axis=-1)
    # shape: (N, n_levels * 4 * d1 + d3)

    # ---- decoder MLP ----
    h = jax.nn.relu(features @ params['W1'] + params['b1'])
    out = h @ params['W2'] + params['b2']   # (N, 4)

    return out

# Training

In [ ]:
def loss_fn(params, key, rays, target, scene_bbox, n_levels, stratified):
    rgb = render_fn_inner(params, key, rays, scene_bbox, n_levels, stratified, False)
    return np.mean(np.square(rgb - target))


def tree_l2_norm(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    return np.sqrt(sum([np.sum(x * x) for x in leaves]))


def train_model(lr, iters, scene_bbox, stratified,
                r1, r2, r3, d1, d2, d3,
                multires_factors, hidden_dim,
                name='', plot_groups=None,
                fg_fraction=0.5, fg_threshold=0.03):
    rng = random.PRNGKey(0)
    n_levels = len(multires_factors)

    # ---- initialise GA-Planes parameters ----
    params = init_gaplanes(rng, r1, r2, r3, d1, d2, d3,
                           multires_factors, hidden_dim)
    n_params = count_params(params)
    print(f'\n[{name}]  total params = {n_params:,}')
    print(f'  multires_factors = {multires_factors}')
    print(f'  base resolutions [r1,r2,r3] = [{r1},{r2},{r3}]')
    print(f'  channel dims     [d1,d2,d3] = [{d1},{d2},{d3}]')
    describe_grids(params, multires_factors, r1, r2, r3, d1, d2, d3)
    print()

    rgb_flat = onp.asarray(training_data[:, 2])
    fg_idx = onp.where(onp.mean(rgb_flat, axis=-1) > fg_threshold)[0]
    bg_idx = onp.where(onp.mean(rgb_flat, axis=-1) <= fg_threshold)[0]
    print(f'foreground rays: {len(fg_idx):,} | background rays: {len(bg_idx):,}')

    opt_init, opt_update, get_params = optimizers.adam(lr)
    opt_state = opt_init(params)

    @jit
    def step_fn(i, opt_state, key, rays, target):
        p = get_params(opt_state)
        loss, g = jax.value_and_grad(loss_fn)(p, key, rays, target,
                                              scene_bbox, n_levels, stratified)
        return opt_update(i, g, opt_state), loss, tree_l2_norm(g)

    if plot_groups is not None:
        plot_groups['PSNR'].append(f'{name}')

    b_i = 0
    xs, psnrs = [], []
    import time
    t = time.time(); t0 = t

    for i in range(iters + 1):
        if fg_fraction is not None and len(fg_idx) > 0 and len(bg_idx) > 0:
            n_fg = int(batch_size * fg_fraction)
            n_bg = batch_size - n_fg
            batch_idx = onp.concatenate([
                onp.random.choice(fg_idx, size=n_fg, replace=True),
                onp.random.choice(bg_idx, size=n_bg, replace=True),
            ])
            onp.random.shuffle(batch_idx)
            batch = training_data[batch_idx]
        else:
            batch = training_data[b_i:b_i + batch_size]
            b_i += batch_size
            if b_i >= training_data.shape[0]:
                b_i = 0

        rays   = np.moveaxis(batch[:, :2], 1, 0)
        target = batch[:, 2]

        rng, key = random.split(rng)
        opt_state, train_loss, grad_norm = step_fn(i, opt_state, key, rays, target)

        if i % 1000 == 0 or i == iters:
            psnr_vals = []
            print(f'{i}  {(time.time()-t)/max(1,i if i<1000 else 1000):.4f} s/iter  '
                  f'{(time.time()-t0)/60:.1f} min total  '
                  f'train_loss={float(train_loss):.5f}  grad_norm={float(grad_norm):.3e}')
            num_vals = val_poses.shape[0] if i == iters else 1
            for v in range(num_vals):
                rays_v = get_rays(H, W, focal, val_poses[v, ...])
                rng, key = random.split(rng)
                rgb, depth, acc = render_fn(
                    get_params(opt_state), key, rays_v, scene_bbox,
                    n_levels, False
                )
                loss_v = np.mean(np.square(rgb - val_images[v, ...]))
                psnr_vals.append(-10.0 * np.log10(loss_v))

            print(f'    render stats: rgb_mean={float(np.mean(rgb)):.4f}, '
                  f'rgb_max={float(np.max(rgb)):.4f}, '
                  f'acc_mean={float(np.mean(acc)):.4f}, '
                  f'acc_max={float(np.max(acc)):.4f}')

            psnr = np.mean(np.array(psnr_vals))
            psnrs.append(psnr)
            xs.append(i)
            if plot_groups is not None:
                plotlosses_model.update({f'{name}': psnr}, current_step=i)
                plotlosses_model.send()
            t = time.time()

    final_params = get_params(opt_state)
    return {
        'state': final_params,
        'psnrs': psnrs,
        'val_image': rgb,
        'xs': xs,
    }


# Run Experiment

In [ ]:
live_plot   = True
reset_plots = True

training_steps = 25000
lr             = 5e-4
batch_size     = 2**10
N_samples      = 512
stratified_sampling = True

multires_factors = [1, 2, 4]

hidden_dim = 128      # decoder hidden width

scene_bbox = (np.array([-1.5, -1.5, -1.5]),
              np.array([ 1.5,  1.5,  1.5]))

# =========================================================================
# varying volume resolution
# r1 = 200               # base resolution for line features
# r2 = 32               # base resolution for plane features
# r3 = -1               # resolution for the volume grid (no multires)
# d1 = 8               # channel dim for lines  (must equal d2)
# d2 = 8               # channel dim for planes (must equal d1)
# d3 = 4               # channel dim for volume
# param_vals = [16, 32]

# varying feature dimension
r1 = 200               # base resolution for line features
r2 = 32               # base resolution for plane features
r3 = 8               # resolution for the volume grid (no multires)
d1 = -1               # channel dim for lines  (must equal d2)
d2 = -1               # channel dim for planes (must equal d1)
d3 = 4               # channel dim for volume
param_vals = [4, 24]

if live_plot:
    if reset_plots:
        plt_groups = {'PSNR': []}
        plotlosses_model = PlotLosses(groups=plt_groups)
else:
    plt_groups = None

if reset_plots:
    outputs_paper = {}

if d1 == -1 and d2 == -1:
    param_to_vary = 'feature_dim'
else:    
    param_to_vary = 'volumeres'
outputs = {}
to_save_outputs = {}
for param_val in param_vals:
    if param_to_vary == 'feature_dim':
        d1 = param_val
        d2 = param_val
        print(f'line_feature_dim: {d1}, plane_feature_dim: {d2}')
    else:
        r3 = param_val
        print(f'{param_to_vary}: {r3}')
    output = train_model(
        lr, training_steps, scene_bbox, stratified_sampling,
        r1, r2, r3, d1, d2, d3,
        multires_factors, hidden_dim,
        name=f'{param_to_vary}={param_val}', plot_groups=plt_groups,
    )
    if param_to_vary == 'feature_dim':
        outputs[f'{d1}'] = output     
    else:
        outputs[f'{r3}'] = output

In [ ]:
outputs_plot = {}
n_levels = len(multires_factors)
for param_val in outputs.keys():
    state = outputs[param_val]['state']
    rng = random.PRNGKey(0)
    rays = get_rays(H, W, focal, test_poses[TEST_IDX])
    rgb, depth, acc = render_fn(state, rng, rays, scene_bbox, n_levels, False)
    rendered = onp.asarray(jax.device_get(rgb))


    outputs_plot[param_val] = {}
    outputs_plot[param_val]['best_pred'] = rgb

with open(f"3d_nerf/gap.pkl", "wb") as f:
    pickle.dump(outputs_plot, f)
    
plot_error_heatmaps(test_images[TEST_IDX,...], outputs_plot, model_name="GA-Planes")